# GeoDiff3D GPU Proof of Concept
Phase 2A: Self-contained remote GPU inference pipeline.

In [ ]:
# Check hardware visibility and VRAM allocation
!nvidia-smi || true


In [ ]:
import os
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
import torch
try:
    import tensorflow as tf
except ImportError:
    tf = None

print('--- Framework Check ---')
print(f'PyTorch CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')

if tf is not None:
    gpu_devices = tf.config.list_physical_devices('GPU')
    print(f'TensorFlow GPUs Available: {len(gpu_devices)}')
else:
    print('TensorFlow GPUs Available: 0 (TensorFlow not installed)')


In [ ]:
import time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
matrix_size = 5000
x_cpu = torch.randn(matrix_size, matrix_size)
y_cpu = torch.randn(matrix_size, matrix_size)

# CPU Time
start = time.time()
_ = torch.matmul(x_cpu, y_cpu)
cpu_time = time.time() - start

# GPU Time
if device.type == 'cuda':
    x_gpu, y_gpu = x_cpu.to(device), y_cpu.to(device)
    _ = torch.matmul(x_gpu, y_gpu) # Warm-up
    torch.cuda.synchronize()
    start = time.time()
    _ = torch.matmul(x_gpu, y_gpu)
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f'Execution complete. Speedup factor: {cpu_time / gpu_time:.1f}x faster on GPU!')


In [ ]:
!pip install diffusers transformers accelerate scipy pillow matplotlib plyfile > /dev/null
!git clone https://github.com/facebookresearch/vggt.git || true
import sys
import os
if os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'vggt')) not in sys.path:
    sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'vggt')))
req_file = '../../vggt/requirements.txt'
if os.path.exists(req_file):
    with open(req_file, 'r') as f:
        lines = f.readlines()
    with open(req_file, 'w') as f:
        for line in lines:
            if 'torch' not in line:  # Strips torch== and torchvision==
                f.write(line)
!pip install -r ../../vggt/requirements.txt > /dev/null || true


In [ ]:
import os
import urllib.request
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)

# Fetch 4 real overlapping images from the classic Middlebury templeRing dataset
urls = [
    'https://raw.githubusercontent.com/alyssaq/3dv/master/dataset/templeRing/templeR0013.png',
    'https://raw.githubusercontent.com/alyssaq/3dv/master/dataset/templeRing/templeR0014.png',
    'https://raw.githubusercontent.com/alyssaq/3dv/master/dataset/templeRing/templeR0015.png',
    'https://raw.githubusercontent.com/alyssaq/3dv/master/dataset/templeRing/templeR0016.png'
]
image_paths = []
import shutil
local_src_dir = None
for p in ['vggt/examples/llff_flower/images', '../../vggt/examples/llff_flower/images']:
    if os.path.exists(p):
        local_src_dir = p
        break
if local_src_dir:
    print('Found local overlapping images in vggt examples. Copying...')
    for i in range(4):
        src_path = os.path.join(local_src_dir, f'{i:03d}.png')
        dst_path = f'input/view_{i:03d}.png'
        shutil.copy(src_path, dst_path)
        image_paths.append(dst_path)
else:
    print('Local images not found. Attempting download...')
    for i, url in enumerate(urls):
        path = f'input/view_{i:03d}.png'
        try:
            urllib.request.urlretrieve(url, path)
            image_paths.append(path)
        except Exception as e:
            print(f'Download failed for {url}: {e}')

print('Downloaded real overlapping images:')
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, path in enumerate(image_paths):
    axes[i].imshow(Image.open(path))
    axes[i].set_title(f'View {i}')
    axes[i].axis('off')
plt.savefig('output/input_grid.png')
plt.show()


In [ ]:
import numpy as np

def align_depth(vggt_depth, diffusion_depth):
    D_d = diffusion_depth.flatten()
    D_v = vggt_depth.flatten()
    valid = (D_d > 0) & (D_v > 0) & (~np.isnan(D_d)) & (~np.isnan(D_v))
    if np.sum(valid) < 10:
        return diffusion_depth, 1.0, 0.0
    D_d_valid = D_d[valid]
    D_v_valid = D_v[valid]
    A = np.vstack([D_d_valid, np.ones(len(D_d_valid))]).T
    a, b = np.linalg.lstsq(A, D_v_valid, rcond=None)[0]
    aligned_depth = a * diffusion_depth + b
    return aligned_depth, a, b

def fuse_depths(vggt_depth, aligned_diffusion_depth, confidence):
    return (1 - confidence) * vggt_depth + confidence * aligned_diffusion_depth

def unproject_to_point_cloud(depth, rgb, K, extrinsic):
    h, w = depth.shape
    u, v = np.meshgrid(np.arange(w), np.arange(h))
    u, v, Z = u.flatten(), v.flatten(), depth.flatten()
    valid = (Z > 0) & (~np.isnan(Z)) & (~np.isinf(Z))
    u, v, Z = u[valid], v[valid], Z[valid]
    colors = rgb.reshape(-1, 3)[valid]
    X_c = (u - K[0, 2]) * Z / K[0, 0]
    Y_c = (v - K[1, 2]) * Z / K[1, 1]
    points_c = np.vstack((X_c, Y_c, Z, np.ones_like(Z)))
    C2W = np.linalg.inv(extrinsic)
    points_w = (C2W @ points_c)[:3, :].T
    return points_w, colors

def save_ply(filename, points, colors):
    header = f'ply\nformat ascii 1.0\nelement vertex {len(points)}\nproperty float x\nproperty float y\nproperty float z\nproperty uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n'
    with open(filename, 'w') as f:
        f.write(header)
        for p, c in zip(points, colors):
            f.write(f'{p[0]:.6f} {p[1]:.6f} {p[2]:.6f} {int(c[0])} {int(c[1])} {int(c[2])}\n')


In [ ]:
import os
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
import torch
import time
import traceback
import numpy as np

print('='*40)
print('VGGT EXECUTION')
print('='*40)

vggt_results = []
cameras = []
vggt_runtime = 0

try:
    start = time.time()
    from vggt.models.vggt import VGGT
    from vggt.utils.load_fn import load_and_preprocess_images
    from vggt.utils.pose_enc import pose_encoding_to_extri_intri
    
    model = VGGT.from_pretrained('facebook/VGGT-1B').to(device)
    model.eval()
    
    images = load_and_preprocess_images(image_paths).to(device)
    
    with torch.no_grad():
        import contextlib
        autocast_context = torch.cuda.amp.autocast() if device.type == 'cuda' else contextlib.nullcontext()
        with autocast_context:
            res = model(images)
        
    extrinsic, intrinsic = pose_encoding_to_extri_intri(res['pose_enc'], images.shape[-2:])
    
    # Move all outputs to numpy, remove batch dimension
    for key in res.keys():
        if isinstance(res[key], torch.Tensor):
            res[key] = res[key].cpu().numpy().squeeze(0)
    extrinsic = extrinsic.cpu().numpy().squeeze(0)
    intrinsic = intrinsic.cpu().numpy().squeeze(0)
    
    vggt_runtime = time.time() - start
    print(f'VGGT executed successfully in {vggt_runtime:.2f}s')
    
    # Extract outputs per view (S, H, W, 1)
    S = res['depth'].shape[0]
    conf_map = res.get('depth_conf', np.ones_like(res['depth']))
    
    for i in range(S):
        vggt_results.append({
            'depth': res['depth'][i].squeeze(-1),
            'confidence': conf_map[i].squeeze(-1) if len(conf_map[i].shape) == 3 else conf_map[i]
        })
        # intrinsic is (S, 3, 3), extrinsic is (S, 4, 4)
        cameras.append({
            'K': intrinsic[i],
            'R': extrinsic[i, :3, :3],
            't': extrinsic[i, :3, 3]
        })
    
except Exception as e:
    print('VGGT execution failed with actual traceback:')
    traceback.print_exc()
    raise RuntimeError('VGGT execution blocked. Please share the traceback above.')


In [ ]:
import os
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
import torch
import time
import traceback

print('='*40)
print('MARIGOLD EXECUTION')
print('='*40)

marigold_results = []
marigold_runtime = 0

try:
    start = time.time()
    from diffusers import MarigoldDepthPipeline
    pipe = MarigoldDepthPipeline.from_pretrained('prs-eth/marigold-depth-lcm-v1-0', torch_dtype=torch.float16).to(device)
    
    for p in image_paths:
        img = Image.open(p).convert('RGB')
        out = pipe(img, num_inference_steps=4)
        marigold_results.append(out.prediction.squeeze())
        
    marigold_runtime = time.time() - start
    print(f'Marigold executed successfully in {marigold_runtime:.2f}s')
    
except Exception as e:
    print('Marigold execution failed with actual traceback:')
    traceback.print_exc()
    raise RuntimeError('Marigold execution blocked. Please share the traceback above.')


In [ ]:
print('='*40)
print('VERIFYING OUTPUTS')
print('='*40)

print('VGGT:')
print(f'runtime = {vggt_runtime:.2f}s')
for i, res in enumerate(vggt_results):
    d = res['depth']
    c = res['confidence']
    print(f'View {i}: depth shape = {d.shape}, dtype = {d.dtype}, min = {d.min():.3f}, max = {d.max():.3f}, NaNs = {np.isnan(d).sum()}, Infs = {np.isinf(d).sum()}')
    print(f'        conf shape = {c.shape}, min = {c.min():.3f}, max = {c.max():.3f}')

print('\nMarigold:')
print(f'runtime = {marigold_runtime:.2f}s')
for i, d in enumerate(marigold_results):
    print(f'View {i}: depth shape = {d.shape}, dtype = {d.dtype}, min = {d.min():.3f}, max = {d.max():.3f}, NaNs = {np.isnan(d).sum()}, Infs = {np.isinf(d).sum()}')


In [ ]:
print('='*40)
print('ALIGNMENT & FUSION')
print('='*40)

aligned_results = []
fused_results = []

for i in range(len(image_paths)):
    v_depth = vggt_results[i]['depth']
    v_conf = vggt_results[i]['confidence']
    m_depth = marigold_results[i]
    
    # Align
    a_depth, scale, shift = align_depth(v_depth, m_depth)
    aligned_results.append(a_depth)
    
    # Fusion (1 - confidence since VGGT confidence implies VGGT is accurate where conf is high)
    f_depth = fuse_depths(v_depth, a_depth, 1.0 - v_conf)
    fused_results.append(f_depth)
    
    print(f'View {i}: Alignment scale={scale:.4f}, shift={shift:.4f}')
    
# Plotting
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for i in range(4):
    axes[0, i].imshow(vggt_results[i]['depth'], cmap='viridis')
    axes[0, i].set_title(f'VGGT Depth {i}')
    axes[1, i].imshow(marigold_results[i], cmap='viridis')
    axes[1, i].set_title(f'Marigold Depth {i}')
    axes[2, i].imshow(aligned_results[i], cmap='viridis')
    axes[2, i].set_title(f'Aligned Marigold {i}')
    axes[3, i].imshow(fused_results[i], cmap='viridis')
    axes[3, i].set_title(f'Fused Depth {i}')
plt.tight_layout()
plt.savefig('output/depth_grids.png')
plt.show()


In [ ]:
print('='*40)
print('3D RECONSTRUCTION')
print('='*40)

b_pts_all, b_cols_all = [], []
g_pts_all, g_cols_all = [], []

for i, p in enumerate(image_paths):
    rgb = np.array(Image.open(p).convert('RGB'))
    cam = cameras[i]
    K = np.array(cam['K'])
    extrinsic = np.vstack([np.hstack([np.array(cam['R']), np.array(cam['t'])]), [0, 0, 0, 1]])
    
    # Baseline
    b_pts, b_cols = unproject_to_point_cloud(vggt_results[i]['depth'], rgb, K, extrinsic)
    b_pts_all.append(b_pts)
    b_cols_all.append(b_cols)
    
    # Guided
    g_pts, g_cols = unproject_to_point_cloud(fused_results[i], rgb, K, extrinsic)
    g_pts_all.append(g_pts)
    g_cols_all.append(g_cols)

save_ply('output/baseline.ply', np.vstack(b_pts_all), np.vstack(b_cols_all))
save_ply('output/guided.ply', np.vstack(g_pts_all), np.vstack(g_cols_all))

print('PLY files written successfully.')
import os
print(f'baseline.ply size: {os.path.getsize("output/baseline.ply")} bytes, points: {sum(len(p) for p in b_pts_all)}')
print(f'guided.ply size: {os.path.getsize("output/guided.ply")} bytes, points: {sum(len(p) for p in g_pts_all)}')
